# Avaliação do RAG — CLT

## Objetivo

Avaliar a qualidade das respostas geradas pelo pipeline RAG (Retrieval-Augmented Generation) sobre a CLT brasileira.

## Metodologia

Utilizamos duas abordagens complementares:

1. **Avaliação automática por palavra-chave (Keyword Accuracy):** verifica se a resposta gerada contém termos-chave esperados (ex.: número do artigo, valor numérico). É rápida e objetiva, mas não captura qualidade semântica.
2. **Análise qualitativa:** comparação manual entre a resposta esperada e a gerada, observando precisão, completude, citação de artigos e clareza.

## Dataset de Avaliação

O arquivo `tests/questions_benchmark.json` contém **10 perguntas** cobrindo os principais temas trabalhistas:
férias, aviso prévio, jornada de trabalho, hora extra, FGTS, licença maternidade,
intervalo intrajornada, rescisão, salário mínimo e trabalho noturno.

---

## 1. Configuração do Ambiente

In [2]:
import json
import sys
from pathlib import Path

import pandas as pd

# Adiciona a raiz do projeto ao sys.path para importar src.*
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.retrieval.chain import get_answer

print(f"Raiz do projeto: {project_root}")
print("Módulos carregados com sucesso.")

Raiz do projeto: C:\Users\teodo\OneDrive\Documentos\Faculdade\6 Semestre Data Science\NLP\clt-rag-chatbot
Módulos carregados com sucesso.


## 2. Carregamento do Benchmark

In [3]:
benchmark_path = project_root / "tests" / "questions_benchmark.json"

with open(benchmark_path, encoding="utf-8") as f:
    benchmark = json.load(f)

print(f"{len(benchmark)} perguntas carregadas.")
print(f"Temas: {', '.join(item['topic'] for item in benchmark)}")

10 perguntas carregadas.
Temas: férias, aviso prévio, jornada de trabalho, hora extra, FGTS, licença maternidade, intervalo intrajornada, rescisão, salário mínimo, trabalho noturno


## 3. Execução dos Testes

Cada pergunta é avaliada de forma isolada (`chat_history=[]`), simulando uma consulta nova ao sistema.

In [4]:
results = []

for item in benchmark:
    # Avaliação isolada: sem histórico de conversa
    generated = get_answer(item["question"], chat_history=[])

    keyword_hit = any(
        kw.lower() in generated.lower() for kw in item["expected_keywords"]
    )

    results.append({
        "id": item["id"],
        "topic": item["topic"],
        "question": item["question"],
        "expected_answer": item["expected_answer"],
        "expected_keywords": item["expected_keywords"],
        "generated_answer": generated,
        "keyword_hit": keyword_hit,
    })

    status = "✅" if keyword_hit else "❌"
    print(f"{status} [{item['topic']:25s}] {item['question'][:60]}...")

✅ [férias                   ] Quantos dias de férias o trabalhador tem direito após 12 mes...
✅ [aviso prévio             ] Qual é o prazo mínimo do aviso prévio na demissão sem justa ...
✅ [jornada de trabalho      ] Qual é a jornada máxima de trabalho diária e semanal previst...
✅ [hora extra               ] Qual o acréscimo mínimo que deve ser pago ao trabalhador por...
✅ [FGTS                     ] Qual é o percentual do FGTS que o empregador deve depositar ...
✅ [licença maternidade      ] Qual é a duração da licença-maternidade prevista na CLT?...
✅ [intervalo intrajornada   ] Qual é o intervalo mínimo para refeição e descanso em jornad...
✅ [rescisão                 ] Quais verbas rescisórias o trabalhador recebe ao ser demitid...
✅ [salário mínimo           ] Como a CLT define o salário mínimo e quais necessidades ele ...
✅ [trabalho noturno         ] Qual é o adicional noturno previsto na CLT e como é contada ...


## 4. Métricas de Avaliação Automática

In [5]:
total = len(results)
hits = sum(r["keyword_hit"] for r in results)
accuracy = hits / total

print(f"Total de perguntas : {total}")
print(f"Acertos (keyword)  : {hits}")
print(f"Erros              : {total - hits}")
print(f"Keyword Accuracy   : {accuracy:.0%}")

Total de perguntas : 10
Acertos (keyword)  : 10
Erros              : 0
Keyword Accuracy   : 100%


In [6]:
df = pd.DataFrame(results)[[
    "id", "topic", "question", "keyword_hit", "expected_keywords"
]]
df["keyword_hit"] = df["keyword_hit"].map({True: "✅", False: "❌"})
df["expected_keywords"] = df["expected_keywords"].apply(", ".join)
df.set_index("id", inplace=True)

pd.set_option("display.max_colwidth", 80)
df

,topic,question,keyword_hit,expected_keywords
id,,,,
1,férias,Quantos dias de férias o trabalhador tem direito após 12 meses de trabalho?,✅,"30 dias, Art. 130"
2,aviso prévio,Qual é o prazo mínimo do aviso prévio na demissão sem justa causa?,✅,"30 dias, Art. 487"
3,jornada de trabalho,Qual é a jornada máxima de trabalho diária e semanal prevista na CLT?,✅,"8 horas, 44 horas, Art. 58"
4,hora extra,Qual o acréscimo mínimo que deve ser pago ao trabalhador por hora extra?,✅,"50%, Art. 59"
5,FGTS,Qual é o percentual do FGTS que o empregador deve depositar mensalmente?,✅,"8%, FGTS"
6,licença maternidade,Qual é a duração da licença-maternidade prevista na CLT?,✅,"120 dias, Art. 392"
7,intervalo intrajornada,Qual é o intervalo mínimo para refeição e descanso em jornadas acima de 6 ho...,✅,"1 hora, Art. 71"
8,rescisão,Quais verbas rescisórias o trabalhador recebe ao ser demitido sem justa causa?,✅,"aviso prévio, férias, FGTS, 40%"
9,salário mínimo,Como a CLT define o salário mínimo e quais necessidades ele deve cobrir?,✅,"Art. 76, salário mínimo"


## 5. Comparação Resposta Esperada × Resposta Gerada

In [7]:
for r in results:
    status = "✅" if r["keyword_hit"] else "❌"
    print(f"{'='*70}")
    print(f"{status} Pergunta #{r['id']} — {r['topic'].upper()}")
    print(f"\nP: {r['question']}")
    print(f"\n[ESPERADO]\n{r['expected_answer']}")
    print(f"\n[GERADO]\n{r['generated_answer']}")
    print()

✅ Pergunta #1 — FÉRIAS

P: Quantos dias de férias o trabalhador tem direito após 12 meses de trabalho?

[ESPERADO]
O trabalhador tem direito a 30 dias corridos de férias após cada período aquisitivo de 12 meses, conforme o Art. 130 da CLT. As férias podem ser divididas em até três períodos, sendo que um deles não pode ser inferior a 14 dias corridos.

[GERADO]
Conforme o Art. 130 da CLT, após cada período de 12 (doze) meses de vigência do contrato de trabalho, o empregado terá direito a férias na seguinte proporção, que varia de acordo com o número de faltas ao serviço:

*   **30 (trinta) dias corridos**, se não houver faltado ao serviço mais de 5 (cinco) vezes (Art. 130, I).
*   **24 (vinte e quatro) dias corridos**, se houver tido de 6 (seis) a 14 (quatorze) faltas (Art. 130, II).
*   **18 (dezoito) dias corridos**, se houver tido de 15 (quinze) a 23 (vinte e três) faltas (Art. 130, III).
*   **12 (doze) dias corridos**, se houver tido de 24 (vinte e quatro) a 32 (trinta e duas) falt

## 6. Análise Qualitativa

Preencha esta seção após executar as células acima, analisando os resultados obtidos.

### 6.1 Pontos Fortes Observados

- *(ex.: o modelo citou corretamente o número do artigo na maioria das perguntas)*
- *(ex.: linguagem clara e acessível, sem jargão excessivo)*
- *(ex.: respostas sobre jornada e hora extra foram precisas e completas)*

### 6.2 Pontos de Melhoria

- *(ex.: perguntas sobre FGTS geraram respostas genéricas sem o percentual correto)*
- *(ex.: o modelo às vezes cita informações que não estão nos trechos recuperados)*
- *(ex.: respostas muito longas para perguntas diretas)*

### 6.3 Casos de Falha

| # | Pergunta | Motivo do Erro |
|---|----------|----------------|
| - | -        | -              |

### 6.4 Hipóteses para os Erros

- **Chunking insuficiente:** o trecho relevante não foi recuperado pelo retriever (k muito baixo ou embedding fraco)
- **Prompt insuficiente:** o modelo não foi guiado a citar o artigo explicitamente
- **Ambiguidade da pergunta:** a pergunta pode ser interpretada de múltiplas formas

### 6.5 Próximos Passos

- [ ] Aumentar `k` no retriever para perguntas com baixa keyword accuracy
- [ ] Avaliar embeddings alternativos (ex.: `text-embedding-004` vs `text-embedding-3-small`)
- [ ] Implementar avaliação semântica com `sentence-transformers` (BLEU, ROUGE ou cosine similarity)
- [ ] Adicionar mais perguntas ao benchmark, incluindo casos-limite e perguntas ambíguas

---

### 6.6 Resultado Final

| Métrica               | Valor |
|-----------------------|-------|
| Keyword Accuracy      | ?/10  |
| Perguntas com artigo  | ?/10  |
| Perguntas completas   | ?/10  |
| Nota qualitativa geral| ? / 5 |

## 7. Avaliação de Relevância dos Chunks

Para cada pergunta, avaliamos se os chunks recuperados pelo retriever são de fato relevantes, usando duas métricas complementares:

| Métrica | Descrição | Vantagem | Limitação |
|---------|-----------|----------|-----------|
| **Cosine similarity** | Distância geométrica entre os embeddings da query e do chunk | Rápido, sem custo de tokens extra | Captura similaridade léxico-semântica, pode trazer falsos positivos |
| **LLM-as-judge** | Gemini avalia o chunk com nota 0–3 | Semântico, entende contexto jurídico | Custa tokens extras (1 chamada por chunk) |
| **Combined** | `0.4 × cosine + 0.6 × llm_score` | Balanceia velocidade e precisão | — |

> **Nota sobre tokens**: a célula abaixo avalia apenas as 3 primeiras perguntas do benchmark (18 chamadas ao Gemini) para economizar cota da API. Remova o `[:3]` para rodar todas as 10.

### 7.1 Scores de relevância por chunk (3 primeiras perguntas)

In [8]:
import pandas as pd
from src.retrieval.retriever import get_retriever
from src.retrieval.relevance import evaluate_chunks

retriever = get_retriever(k=6)

# Avalia relevância dos chunks para as 3 primeiras perguntas (economiza tokens)
relevance_results = []

for item in benchmark[:3]:
    docs = retriever.invoke(item["question"])
    scored = evaluate_chunks(item["question"], docs)

    for rank, s in enumerate(scored, start=1):
        relevance_results.append({
            "pergunta_id": item["id"],
            "topic": item["topic"],
            "rank": rank,
            "artigo": s["doc"].metadata.get("artigo", "—"),
            "cosine": s["cosine"],
            "llm_score": s["llm_score"],
            "combined": s["combined"],
            "trecho": s["doc"].page_content[:80] + "...",
        })

df_rel = pd.DataFrame(relevance_results)
pd.set_option("display.max_colwidth", 85)
df_rel

,pergunta_id,topic,rank,artigo,cosine,llm_score,combined,trecho
0,1,férias,1,Art. 130,0.692,0.0,0.277,Art. 130. Após cada período de 12 (doze) meses de vigência do contrato de trabal...
1,1,férias,2,Art. 140,0.649,0.0,0.259,"Art. 140. Os empregados contratados há menos de 12 (doze) meses gozarão, na opor..."
2,1,férias,3,Art. 146,0.635,0.0,0.254,"Art. 146. Na cessação do contrato de trabalho, qualquer que seja a sua causa, se..."
3,1,férias,4,Art. 17,0.633,0.0,0.253,Art. 17. O empregado doméstico terá direito a férias anuais remuneradas de 30 (t...
4,1,férias,5,Art. 134,0.630,0.0,0.252,"Art. 134. As férias serão concedidas por ato do empregador, em um só período, no..."
5,1,férias,6,Art. 129,0.625,0.0,0.250,Art. 129. Todo empregado terá direito anualmente ao gozo de um período de férias...
6,2,aviso prévio,1,Art. 487,0.714,0.0,0.285,"Art. 487. Não havendo prazo estipulado, a parte que, sem justo motivo, quiser re..."
7,2,aviso prévio,2,Art. 23,0.688,0.0,0.275,"Art. 23. Não havendo prazo estipulado no contrato, a parte que, sem justo motivo..."
8,2,aviso prévio,3,Art. 490,0.661,0.0,0.264,"Art. 490. O empregador que, durante o prazo do aviso prévio dado ao empregado, p..."
9,2,aviso prévio,4,Art. 491,0.654,0.0,0.261,"Art. 491. O empregado que, durante o prazo do aviso prévio, cometer qualquer das..."


## 8. Avaliação de Recall

**Recall@k** mede se o artigo esperado para cada pergunta está entre os k chunks recuperados pelo retriever.

- **Hit**: o artigo esperado (ex: `Art. 130`) aparece no metadata de pelo menos um dos k chunks retornados
- **Miss**: o artigo esperado não foi recuperado — o pipeline nunca terá a informação correta, independente do LLM

Esta é a métrica mais crítica do RAG: um LLM perfeito não consegue responder corretamente se o retriever não trouxe o chunk certo.

In [9]:
import pandas as pd
from src.retrieval.retriever import get_retriever
from src.retrieval.relevance import score_cosine, score_llm

retriever = get_retriever(k=6)

# Avalia recall para todas as perguntas do benchmark
recall_results = []

for item in benchmark:
    docs = retriever.invoke(item["question"])
    retrieved_articles = [d.metadata.get("artigo", "") for d in docs]

    # Extrai artigos esperados das keywords (ex: "Art. 130")
    expected_arts = [kw for kw in item["expected_keywords"] if kw.startswith("Art.")]
    hit = any(art in retrieved_articles for art in expected_arts) if expected_arts else None

    recall_results.append({
        "id": item["id"],
        "topic": item["topic"],
        "expected_articles": ", ".join(expected_arts) if expected_arts else "—",
        "retrieved_articles": ", ".join(retrieved_articles[:4]) + ("..." if len(retrieved_articles) > 4 else ""),
        "recall_hit": "✅" if hit else ("❌" if hit is False else "N/A"),
    })

df_recall = pd.DataFrame(recall_results).set_index("id")
pd.set_option("display.max_colwidth", 60)

total_with_art = sum(1 for r in recall_results if r["recall_hit"] != "N/A")
hits = sum(1 for r in recall_results if r["recall_hit"] == "✅")
print(f"Recall@6 (artigo esperado entre os 6 chunks recuperados): {hits}/{total_with_art} = {hits/total_with_art:.0%}\n")
df_recall

Recall@6 (artigo esperado entre os 6 chunks recuperados): 8/8 = 100%



,topic,expected_articles,retrieved_articles,recall_hit
id,,,,
1,férias,Art. 130,"Art. 130, Art. 140, Art. 146, Art. 17...",✅
2,aviso prévio,Art. 487,"Art. 487, Art. 23, Art. 490, Art. 491...",✅
3,jornada de trabalho,Art. 58,"Art. 58, Art. 432, Art. 2º, Art. 293...",✅
4,hora extra,Art. 59,"Art. 59, Art. 296, Art. 305, Art. 241...",✅
5,FGTS,—,"Art. 34, Art. 21, Art. 22, Art. 14-A...",N/A
6,licença maternidade,Art. 392,"Art. 392, Art. 25, Art. 392-A, Art. 392-B...",✅
7,intervalo intrajornada,Art. 71,"Art. 5º, Art. 71, Art. 66, Art. 244...",✅
8,rescisão,—,"Art. 6º, Art. 147, Art. 479, Art. 477...",N/A
9,salário mínimo,Art. 76,"Art. 76, Art. 81, Art. 7º, Art. 458...",✅


## 9. RAG Agêntico vs RAG Padrão

O RAG agêntico implementado em `src/retrieval/agent.py` estende o pipeline padrão com um **loop de avaliação e retry**:

| Etapa | RAG Padrão | RAG Agêntico |
|-------|-----------|--------------|
| Retrieval | k=6, uma vez | k=8, até 3 tentativas |
| Avaliação dos chunks | ✗ | ✅ cosine + LLM-as-judge |
| Reformulação da query | ✗ | ✅ se relevância média < 0.45 |
| Filtragem de chunks irrelevantes | ✗ | ✅ apenas chunks acima do threshold |
| Custo por pergunta (tokens) | ~800 | ~3.000–8.000 (depende das tentativas) |

**Quando o agente reformula a query?**
Se a média dos scores combinados (`0.4 × cosine + 0.6 × llm_judge`) for abaixo de `0.45`, o agente pede ao Gemini para reescrever a query com termos mais específicos da CLT, e repete o retrieval.

**Quando vale o custo extra?**
- Perguntas ambíguas ou com vocabulário informal ("meu chefe me mandou embora, o que recebo?")
- Perguntas que cruzam múltiplos artigos
- Casos onde o RAG padrão retornou resposta vaga ("não encontrei na CLT")

### 9.1 Exemplo de execução

In [10]:
# Exemplo de uso do RAG agêntico
# (execute após as seções 7 e 8 para comparação)
from src.retrieval.agent import get_agent_answer

sample_question = benchmark[0]["question"]
print(f"Pergunta: {sample_question}\n")
print("Executando RAG agêntico (pode demorar — faz múltiplas chamadas)...")

agent_answer = get_agent_answer(sample_question, chat_history=[])
print(f"\nResposta agêntica:\n{agent_answer}")

c:\Users\teodo\OneDrive\Documentos\Faculdade\6 Semestre Data Science\NLP\clt-rag-chatbot\.venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Pergunta: Quantos dias de férias o trabalhador tem direito após 12 meses de trabalho?

Executando RAG agêntico (pode demorar — faz múltiplas chamadas)...

Resposta agêntica:
Conforme o Art. 130 da CLT, após cada período de 12 (doze) meses de vigência do contrato de trabalho, o empregado terá direito a férias na seguinte proporção, que varia de acordo com o número de faltas ao serviço:

*   **30 (trinta) dias corridos**, se não houver faltado ao serviço mais de 5 (cinco) vezes.
*   **24 (vinte e quatro) dias corridos**, se houver tido de 6 (seis) a 14 (quatorze) faltas.
*   **18 (dezoito) dias corridos**, se houver tido de 15 (quinze) a 23 (vinte e três) faltas.
*   **12 (doze) dias corridos**, se houver tido de 24 (vinte e quatro) a 32 (trinta e duas) faltas.

É importante notar que, conforme o § 1º do Art. 130, é vedado descontar as faltas do empregado ao serviço do período de férias.
